In [0]:
# Databricks notebook source
# ==============================================================================
# PHASE 9: Gold Layer Aggregation & Dimensional Modeling
# ==============================================================================

storage_account_name = "datalakeseniorproj012026"
kv_scope_name = "kv-portfolio12026"

print("1. Authenticating & Injecting OAuth 2.0 configuration...")
client_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-id")
tenant_id = dbutils.secrets.get(scope=kv_scope_name, key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope=kv_scope_name, key="sp-client-secret")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

silver_uri = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/"
gold_uri = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/"

print("2. Reading cleaned tables from Silver Delta Lake...")
df_silver_lineitem = spark.read.format("delta").load(silver_uri + "lineitem/")
df_silver_orders = spark.read.format("delta").load(silver_uri + "orders/")

print("3. Building Gold Business Aggregate (Sales Summary by Order Status)...")
from pyspark.sql.functions import col, sum, count, round

# Join orders and lineitems to create a conformed fact table summary
df_gold_sales = df_silver_orders.join(
    df_silver_lineitem,
    df_silver_orders.o_orderkey == df_silver_lineitem.l_orderkey,
    "inner"
).groupBy("o_orderstatus").agg(
    count("o_orderkey").alias("total_orders"),
    round(sum("l_extendedprice"), 2).alias("total_revenue")
)

print("4. Persisting Gold aggregate table to Delta Lake...")
df_gold_sales.write.format("delta").mode("overwrite").save(gold_uri + "sales_summary/")

print("===============================================================================")
print("SUCCESS: Gold layer analytical summary created and persisted successfully.")
print("===============================================================================")

display(dbutils.fs.ls(gold_uri))